# Root Water Content Visualization

This notebook installs the required dependencies, loads the cleaned workbook, prepares the root water content data, and builds interactive Plotly charts.

In [ ]:
%pip install pandas plotly numpy openpyxl kaleido nbformat scipy

## 1. Install and Import Dependencies

Run the next cell to make sure the notebook kernel has the required packages. The import cell also configures Plotly for notebook rendering.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from scipy import stats

pio.renderers.default = "notebook_connected"

## 2. Create or Load Root Water Content Data

This notebook loads the cleaned Excel workbook created in the project folder. If you later replace the file with a CSV, the same preparation logic can be adapted easily.

In [2]:
workbook_path = Path("fungal_mycorrhizal_tropical new_filtered_clean.xlsx")

if not workbook_path.exists():
    raise FileNotFoundError(f"Expected workbook at {workbook_path.resolve()}")

df = pd.read_excel(workbook_path)
df.head(10)

,Sample,Gen_sp,MycorrhizalType,RootWaterContent_pct,RootVolume_cm3
0,1,Acer laurinum,AM,68.478284,0.627933
1,2,Acronychia pedunculata,AM,73.465447,0.583444
2,3,Actinodaphne henryi,AM,68.423235,0.529667
3,4,Alchornea tiliifolia,AM,72.278069,0.448556
4,5,Antiaris toxicaria,AM,72.566125,0.447000
5,6,Aporusa yunnanensis,AM,70.946875,0.280667
6,7,Baccaurea ramiflora,AM,76.124202,0.351444
7,8,Barringtonia fusicarpa,AM,71.512218,0.672167
8,9,Beilschmiedia percoriacea,AM,64.741122,0.422667
9,10,Canarium album,AM,73.788503,0.236444


## 3. Prepare Data for Plotly

The next cell enforces numeric types, removes incomplete rows, flags outliers with the 1.5×IQR rule on both plotted axes, and keeps the filtered visuals stable and readable.

In [3]:
required_columns = [
    "Sample",
    "Gen_sp",
    "MycorrhizalType",
    "RootWaterContent_pct",
    "RootVolume_cm3",
]

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

plot_df = df.loc[:, required_columns].copy()
plot_df["Sample"] = pd.to_numeric(plot_df["Sample"], errors="coerce")
plot_df["RootWaterContent_pct"] = pd.to_numeric(
    plot_df["RootWaterContent_pct"], errors="coerce"
)
plot_df["RootVolume_cm3"] = pd.to_numeric(
    plot_df["RootVolume_cm3"], errors="coerce"
)
plot_df["Gen_sp"] = plot_df["Gen_sp"].astype("string")
plot_df["MycorrhizalType"] = plot_df["MycorrhizalType"].astype("string")

plot_df = plot_df.dropna(
    subset=["Sample", "RootWaterContent_pct", "RootVolume_cm3"]
).copy()

outlier_columns = ["RootWaterContent_pct", "RootVolume_cm3"]
outlier_mask = pd.Series(False, index=plot_df.index)
outlier_bounds = {}

for column in outlier_columns:
    q1 = plot_df[column].quantile(0.25)
    q3 = plot_df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_bounds[column] = (lower_bound, upper_bound)
    outlier_mask |= plot_df[column].lt(lower_bound) | plot_df[column].gt(upper_bound)

outlier_df = plot_df.loc[outlier_mask].sort_values(["MycorrhizalType", "Sample"]).copy()
plot_df = plot_df.loc[~outlier_mask].sort_values(
    ["MycorrhizalType", "RootVolume_cm3"]
).reset_index(drop=True)

plot_df["RootWaterContent_zscore"] = (
    plot_df["RootWaterContent_pct"] - plot_df["RootWaterContent_pct"].mean()
) / plot_df["RootWaterContent_pct"].std(ddof=0)

pvalue_records = []
for mycorrhizal_type, group_df in plot_df.groupby("MycorrhizalType", dropna=False):
    group_values = group_df["RootWaterContent_pct"].dropna()
    other_values = plot_df.loc[
        plot_df["MycorrhizalType"] != mycorrhizal_type, "RootWaterContent_pct"
    ].dropna()

    if len(group_values) >= 2 and len(other_values) >= 2:
        test_result = stats.ttest_ind(
            group_values, other_values, equal_var=False, nan_policy="omit"
        )
        p_value = float(test_result.pvalue)
    else:
        p_value = np.nan

    pvalue_records.append(
        {
            "MycorrhizalType": mycorrhizal_type,
            "n": len(group_values),
            "p_value": p_value,
        }
    )

pvalue_df = pd.DataFrame(pvalue_records).sort_values("MycorrhizalType").reset_index(
    drop=True
)
pvalue_annotation = "<br>".join(
    f"{row.MycorrhizalType}: p={row.p_value:.4g} (n={int(row.n)})"
    if pd.notna(row.p_value)
    else f"{row.MycorrhizalType}: p=NA (n={int(row.n)})"
    for row in pvalue_df.itertuples()
 )

print(
    f"Removed {len(outlier_df)} outlier rows using the 1.5×IQR rule across RootWaterContent_pct and RootVolume_cm3."
)
display(
    pvalue_df.assign(
        p_value=pvalue_df["p_value"].map(
            lambda value: f"{value:.4g}" if pd.notna(value) else "NA"
        )
    )
)
outlier_df[[
    "Sample",
    "Gen_sp",
    "MycorrhizalType",
    "RootWaterContent_pct",
    "RootVolume_cm3",
]]

Removed 10 outlier rows using the 1.5×IQR rule across RootWaterContent_pct and RootVolume_cm3.


,MycorrhizalType,n,p_value
0,AM,46,0.002749
1,ECM,10,0.002749


,Sample,Gen_sp,MycorrhizalType,RootWaterContent_pct,RootVolume_cm3
19,20,Cryptocarya yunnanensis,AM,70.790226,1.280333e+09
23,24,Ficus langkokensis,AM,71.467178,1.004167e+09
27,28,Gironniera subaequalis,AM,7.156791,3.606667e-01
31,32,Horsfieldia amygdalina,AM,74.350278,1.769750e+05
32,33,Horsfieldia kingii,AM,76.962706,2.688000e+03
33,34,Knema tenuinervia,AM,70.841618,1.021000e+03
36,37,Litsea elongata,AM,7.091922,4.665000e-01
42,43,Machilus nanmu,AM,72.257826,1.100333e+09
63,64,Castanopsis ferox,ECM,6.723796,5.231667e-01
64,65,Castanopsis hystrix,ECM,6.597020,4.046190e-01


## 4. Plot Root Water Content Against Root Volume

This chart uses the filtered dataset after IQR-based outlier removal, with `RootVolume_cm3` on the x-axis and root water content on the y-axis. The annotation shows Welch-test p-values for each mycorrhizal type versus the remaining filtered data.

In [6]:
sample_figure = px.line(
    plot_df,
    x="RootVolume_cm3",
    y="RootWaterContent_pct",
    color="MycorrhizalType",
    markers=True,
    hover_data={
        "Sample": True,
        "Gen_sp": True,
        "RootVolume_cm3": ":.4f",
        "RootWaterContent_zscore": ":.2f",
    },
    title="Root Water Content by Root Volume",
    labels={
        "RootVolume_cm3": "Root Volume (cm^3)",
        "RootWaterContent_pct": "Root Water Content (%)",
        "MycorrhizalType": "Mycorrhizal Type",
    },
)

sample_figure.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Mycorrhizal Type",
    legend={
        "x": 1.02,
        "y": 1,
        "xanchor": "left",
        "yanchor": "top",
    },
    margin={"r": 220},
)
sample_figure.add_annotation(
    xref="paper",
    yref="paper",
    x=1.02,
    y=0.02,
    xanchor="left",
    yanchor="bottom",
    showarrow=False,
    align="left",
    text="Welch t-test p-values<br>vs remaining types<br><br>" + pvalue_annotation,
    bgcolor="rgba(255, 255, 255, 0.9)",
    bordercolor="rgba(0, 0, 0, 0.25)",
    borderwidth=1,
)

sample_figure.show()

## 5. Add Interactive Plot Controls

The chart below keeps the group comparison view and repeats the same per-type p-value annotation for consistency.

In [7]:
box_figure = px.box(
    plot_df,
    x="MycorrhizalType",
    y="RootWaterContent_pct",
    color="MycorrhizalType",
    points="all",
    hover_data={
        "Sample": True,
        "Gen_sp": True,
        "RootVolume_cm3": ":.4f",
    },
    title="Distribution of Root Water Content by Mycorrhizal Type",
    labels={
        "MycorrhizalType": "Mycorrhizal Type",
        "RootWaterContent_pct": "Root Water Content (%)",
    },
)

box_figure.update_layout(
    template="plotly_white",
    showlegend=False,
    margin={"r": 220},
)
box_figure.add_annotation(
    xref="paper",
    yref="paper",
    x=1.02,
    y=0.02,
    xanchor="left",
    yanchor="bottom",
    showarrow=False,
    align="left",
    text="Welch t-test p-values<br>vs remaining types<br><br>" + pvalue_annotation,
    bgcolor="rgba(255, 255, 255, 0.9)",
    bordercolor="rgba(0, 0, 0, 0.25)",
    borderwidth=1,
)

box_figure.show()

## 6. Export or Save the Figure

The next cell saves the interactive Plotly chart as HTML and attempts a static PNG export when the image engine is available.

In [ ]:
output_html = Path("root_water_content_plot.html")
output_png = Path("root_water_content_plot.png")

sample_figure.write_html(output_html, include_plotlyjs="cdn")
print(f"Saved interactive chart to {output_html.resolve()}")

try:
    sample_figure.write_image(output_png, width=1200, height=700, scale=2)
    print(f"Saved static chart to {output_png.resolve()}")
except Exception as error:
    print("Static export skipped. Install or update kaleido if needed.")
    print(error)